# 06 — Phishing Detection and Social Engineering Analysis

## What This Is
Phishing attacks use deceptive communications to steal credentials, install malware, or manipulate behaviour. NLP-based detectors analyse email content, URL structure, and sender metadata to classify phishing attempts.

## Why It Matters
Verizon DBIR consistently ranks phishing as the top initial access vector. Spear phishing (targeted), whaling (executives), and smishing (SMS) cause billions in losses annually. Defenders build classifiers to filter these at scale.

## Where the Community Stands
Modern email gateways (Proofpoint, Mimecast) combine URL reputation, DMARC/SPF validation, ML classifiers, and sandboxed attachment analysis. LLMs are emerging as both phishing generators (red teams) and detectors.

## Authorized Testing Context
All phishing analysis here is purely defensive — building detection systems. This is not a phishing kit. Red team phishing simulations require explicit written authorisation from organisational leadership.

In [ ]:
import re
import math
import hashlib
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from urllib.parse import urlparse

# --- Feature extraction from email/URL ---

URGENT_WORDS = [
    'urgent', 'immediate', 'action required', 'verify now', 'suspended',
    'confirm your', 'click here', 'limited time', 'expires', 'act now',
    'your account', 'unauthorized access', 'security alert', 'winner',
    'free gift', 'congratulations', 'invoice attached', 'final notice'
]

TRUSTED_DOMAINS = {
    'gmail.com', 'yahoo.com', 'outlook.com', 'microsoft.com',
    'apple.com', 'amazon.com', 'paypal.com', 'bankofamerica.com'
}

SUSPICIOUS_TLDS = {'.tk', '.ml', '.ga', '.cf', '.gq', '.xyz', '.top', '.click', '.loan'}

def extract_email_features(email: Dict) -> Dict:
    """Extract phishing-indicative features from an email dict."""
    subject  = email.get('subject', '').lower()
    body     = email.get('body', '').lower()
    sender   = email.get('from', '').lower()
    urls     = email.get('urls', [])
    text     = subject + ' ' + body

    # Urgency signal
    urgency_count = sum(1 for w in URGENT_WORDS if w in text)

    # Sender domain analysis
    sender_domain = ''
    m = re.search(r'@([\w\.-]+)', sender)
    if m:
        sender_domain = m.group(1)
    domain_trusted    = sender_domain in TRUSTED_DOMAINS
    domain_lookalike  = any(
        (td in sender_domain and td != sender_domain)
        for td in TRUSTED_DOMAINS
    )

    # URL features
    suspicious_urls = 0
    ip_urls         = 0
    long_urls       = 0
    for url in urls:
        try:
            parsed = urlparse(url)
            netloc = parsed.netloc.lower()
            if re.match(r'^\d+\.\d+\.\d+\.\d+', netloc):
                ip_urls += 1
            if any(netloc.endswith(tld) for tld in SUSPICIOUS_TLDS):
                suspicious_urls += 1
            if len(url) > 100:
                long_urls += 1
        except:
            pass

    # Text statistics
    word_count  = len(text.split())
    exclamation = text.count('!')
    caps_ratio  = sum(1 for c in (subject+body) if c.isupper()) / max(1, len(subject+body))

    return {
        'urgency_count':   urgency_count,
        'domain_trusted':  int(domain_trusted),
        'domain_lookalike':int(domain_lookalike),
        'suspicious_urls': suspicious_urls,
        'ip_urls':         ip_urls,
        'long_urls':       long_urls,
        'word_count':      word_count,
        'exclamations':    exclamation,
        'caps_ratio':      round(caps_ratio, 3),
        'url_count':       len(urls),
    }

print('Feature extraction defined')

In [ ]:
def phishing_score(features: Dict) -> Tuple[float, str]:
    """Rule-based phishing probability (0-1)."""
    score = 0.0

    score += min(0.30, features['urgency_count']   * 0.08)
    score += features['domain_lookalike']          * 0.25
    score += features['suspicious_urls']           * 0.15
    score += features['ip_urls']                   * 0.20
    score += min(0.10, features['long_urls']       * 0.05)
    score += min(0.10, features['exclamations']    * 0.03)
    score += min(0.10, features['caps_ratio']      * 0.30)
    if features['domain_trusted']:
        score -= 0.15  # slight trust boost (can still be spoofed)

    score = max(0.0, min(1.0, score))
    if score >= 0.70:   label = 'PHISHING'
    elif score >= 0.40: label = 'SUSPICIOUS'
    else:               label = 'CLEAN'
    return round(score, 3), label

# Test emails
EMAILS = [
    {
        'id': 'E001',
        'from': 'security@paypa1.com',
        'subject': 'URGENT: Your PayPal account has been SUSPENDED!',
        'body': 'Click here immediately to verify your account or it will be closed. Action required now!',
        'urls': ['http://203.0.113.42/paypal/verify?token=abc123longtoken'*3],
    },
    {
        'id': 'E002',
        'from': 'newsletter@legitimate-company.com',
        'subject': 'Your monthly newsletter',
        'body': 'Here are the top articles from this month. Read more on our website.',
        'urls': ['https://legitimate-company.com/newsletter/2024/03'],
    },
    {
        'id': 'E003',
        'from': 'hr@corp-internal.xyz',
        'subject': 'Invoice #4821 attached - FINAL NOTICE',
        'body': 'Please review and pay the attached invoice to avoid service interruption. Expires TODAY!',
        'urls': ['http://evil.tk/invoice/pay'],
    },
    {
        'id': 'E004',
        'from': 'alice@gmail.com',
        'subject': 'Lunch plans for tomorrow?',
        'body': 'Hey, are you free for lunch tomorrow around noon? Let me know!',
        'urls': [],
    },
]

print(f'{"ID":<6} {"Score":<8} {"Label":<12} {"Urgency":<10} {"Lookalike":<12} {"Susp URLs"}')
print('-' * 70)
for email in EMAILS:
    feats = extract_email_features(email)
    score, label = phishing_score(feats)
    print(f'{email["id"]:<6} {score:<8} {label:<12} {feats["urgency_count"]:<10} {feats["domain_lookalike"]:<12} {feats["suspicious_urls"]}')

## URL Reputation and Typosquatting Detection
Typosquatting registers domains that visually resemble trusted brands (paypa1.com, arnazon.com). Edit-distance metrics catch these patterns.

In [ ]:
def levenshtein(s1: str, s2: str) -> int:
    m, n = len(s1), len(s2)
    dp = list(range(n+1))
    for i in range(1, m+1):
        prev = dp.copy()
        dp[0] = i
        for j in range(1, n+1):
            if s1[i-1] == s2[j-1]:
                dp[j] = prev[j-1]
            else:
                dp[j] = 1 + min(prev[j], dp[j-1], prev[j-1])
    return dp[n]

def typosquat_check(domain: str, trusted_domains: List[str], threshold: int = 2) -> List[Tuple[str, int]]:
    """Return (trusted_domain, edit_distance) pairs within threshold."""
    results = []
    # Strip TLD for comparison
    d_core = domain.split('.')[0]
    for td in trusted_domains:
        t_core = td.split('.')[0]
        dist = levenshtein(d_core, t_core)
        if 0 < dist <= threshold:
            results.append((td, dist))
    return sorted(results, key=lambda x: x[1])

test_domains = [
    'paypa1.com', 'arnazon.com', 'micros0ft.com', 'gooogle.com',
    'netflix.com', 'faceb00k.com', 'gmail.com', 'apple-support.xyz'
]

TRUSTED = list(TRUSTED_DOMAINS)
print(f'{"Domain":<25} {"Potential Typosquat of":<25} {"Edit Dist"}')
print('-' * 60)
for d in test_domains:
    hits = typosquat_check(d, TRUSTED)
    if hits:
        for td, dist in hits[:1]:
            print(f'{d:<25} {td:<25} {dist}')
    else:
        print(f'{d:<25} {"(no match)":<25}')

## Defender Playbook
A phishing detection pipeline combines URL reputation, email authentication (SPF/DKIM/DMARC), content classification, and sandboxed attachment analysis. No single signal is sufficient — stack them.

In [ ]:
def defender_playbook(email: Dict) -> Dict:
    feats        = extract_email_features(email)
    score, label = phishing_score(feats)

    # URL typosquat check
    typo_hits = []
    for url in email.get('urls', []):
        try:
            netloc = urlparse(url).netloc.split(':')[0]
            hits   = typosquat_check(netloc, list(TRUSTED_DOMAINS))
            typo_hits.extend(hits)
        except:
            pass

    # Sender domain
    m = re.search(r'@([\w\.-]+)', email.get('from',''))
    sender_domain = m.group(1) if m else ''
    sender_typos  = typosquat_check(sender_domain, list(TRUSTED_DOMAINS))

    action = 'QUARANTINE' if label == 'PHISHING' else ('REVIEW' if label == 'SUSPICIOUS' else 'DELIVER')

    return {
        'email_id':      email.get('id', '?'),
        'phish_score':   score,
        'classification':label,
        'action':        action,
        'sender_typo':   sender_typos,
        'url_typo':      typo_hits,
        'key_signals':   {k:v for k,v in feats.items() if v != 0},
    }

for email in EMAILS:
    result = defender_playbook(email)
    print(f"\n[{result['email_id']}] {result['classification']} (score={result['phish_score']}) -> {result['action']}")
    if result['sender_typo']:
        print(f"  Sender typosquat: {result['sender_typo']}")
    if result['url_typo']:
        print(f"  URL typosquat:    {result['url_typo']}")